# 3. 切り替えアプリのデプロイ（任意）

このノートブックは、1-2（Scratch）と 2（Managed）で作った2つのエージェントを、1つのチャット画面で切り替えて比べるアプリ `apps/rag-handson-switcher` をデプロイするガイドである。**Databricks の画面の操作だけで行う**。ローカル PC や Web Terminal は使わない。コードのセルはない。各 STEP の画像の赤枠と番号の順に、画面を操作する。

```
Git フォルダー（apps/rag-handson-switcher）── デプロイ ──→ Databricks Apps（AppKit のチャット）
                                                          ├─ Scratch タブ：自分の AI Search 索引を検索 → LLM が回答
                                                          └─ Managed タブ：Genie Agent（Agent mode）が回答
```

## このノートブックでやること
1. Databricks Apps の画面でアプリを作る
2. アプリのリソースに自分の索引を、ユーザー認証のスコープを3つ追加する
3. Git フォルダーのアプリのフォルダーからデプロイする
4. Managed タブに Genie Agent を指定し、同じ質問を2つのエージェントに投げて比べる

## 前提
- このリポジトリを Git フォルダーとして取り込んでいる（README の「ノートブックのインポート」）
- **1-1** で自分の AI Search 索引を作っている
- **2** で Genie Agent を作っている
- Databricks Apps を作る権限がある

## アプリの仕組み
どちらのタブも、**質問した人本人の権限**（ユーザー認証）で呼ぶ。アプリのサービスプリンシパルに索引や Genie Agent の権限を付ける必要はない。

| タブ | 呼ぶもの | 使うスコープ |
|---|---|---|
| Scratch | 1-1 の索引をハイブリッド検索し、1-2 のエージェントと同じシステムプロンプト・同じ LLM で回答を作る | `vector-search`、`model-serving` |
| Managed | Genie Agent の Agent mode API | `genie` |

アプリの設定は、アプリのフォルダーの `app.yaml` にある。索引の名前は、STEP 7 で追加するリソース（リソースキー `vector-search-index`）から受け取るので、ファイルを編集する必要はない。Genie Agent は、STEP 13 でアプリの画面に URL を貼って指定する。

> Scratch タブは、1-2 でデプロイしたエージェントのアプリを呼ばずに、同じ索引と同じ LLM を直接呼ぶ。アプリから別のアプリを呼ぶと、利用者のトークンが引き継がれないためである。

画像はこのノートブックと同じフォルダーの `images/` にある。

---

## 3-1. アプリを作る

### STEP 1：Databricks Apps を開く
①ワークスペースの右上の「アプリを切り替え」（点が9つのアイコン）をクリックし、②「Databricks Apps」をクリックする。

![STEP 1](./images/switcher-01.png)


→ Databricks Apps の画面が開く。「コンピュート」→「Apps」タブの「Use the app switcher」からも開ける。

### STEP 2：アプリの作成を始める
左の「アプリ」で、①右の「アプリを作成」をクリックする。

![STEP 2](./images/switcher-02.png)


→ 「新しいアプリを作成」の画面が開く。

### STEP 3：カスタムアプリを選ぶ
①「カスタムアプリを作成」をクリックする。

![STEP 3](./images/switcher-03.png)


→ アプリの作成が3つの手順（アプリ名、Git リポジトリ、構成）で進む。

### STEP 4：アプリの名前を付ける
①「アプリ名」に、ほかの参加者と重ならない名前（例 `switcher-<自分の名前>`、英小文字・数字・ハイフン）を入れ、②「次へ：Gitを設定」をクリックする。

![STEP 4](./images/switcher-04.png)


→ 名前はあとから変えられない。

### STEP 5：Git リポジトリの設定は飛ばす
何も入れずに、①「次へ：構成」をクリックする。

![STEP 5](./images/switcher-05.png)


→ コードは、あとで Git フォルダーからデプロイする（手順 11）。

> **注意**：このリポジトリは private のため、ここで Git リポジトリを設定して直接デプロイするには、アプリのサービスプリンシパルに GitHub の資格情報を設定する必要がある。

## 3-2. 索引とスコープを設定する

### STEP 6：リソースに AI Search の索引を追加する
「アプリのリソース」で、①「リソースを追加」をクリックし、②「AI検索インデックス」を選ぶ。

![STEP 6](./images/switcher-06.png)


→ AI Search の索引を指定する行が追加される。

### STEP 7：自分の索引を指定する
①1-1 で作った自分の索引（`<カタログ>.<自分のスキーマ>.docs_chunks_index`）を入力する（フォルダーのアイコンから選んでもよい）。②権限は「選択可能」、③リソースキーは `vector-search-index` のままにする。

![STEP 7](./images/switcher-07.png)


→ アプリは、`app.yaml` でこのリソースキーから索引の名前を受け取る。

> **注意**：リソースキーを変えると、アプリが索引を見つけられない。

### STEP 8：ユーザー認証のスコープを追加する
「ユーザー認証」で、①「スコープを追加」をクリックし、`genie`、「モデルサービング」（`model-serving`）、「ベクトル検索」（`vector-search`）を1つずつ追加する。②3つが一覧に並んだことを確かめる。

![STEP 8](./images/switcher-08.png)


→ アプリは、この3つのスコープで、質問した人本人の権限で Genie Agent・LLM・AI Search を呼ぶ。

### STEP 9：アプリを作成する
画面の下までスクロールし、①「アプリを作成」をクリックする。

![STEP 9](./images/switcher-09.png)

## 3-3. Git フォルダーからデプロイする

### STEP 10：コンピュートの起動を待つ
①「コンピュートを開始しています」が消えるまで 2〜3分待つ。

![STEP 10](./images/switcher-10.png)


→ 起動中は「デプロイ」をクリックできない。

### STEP 11：Git フォルダーからデプロイする
右上の「デプロイ」をクリックする。①「ソースコードのパス」に、Git フォルダーの中のアプリのフォルダー（例 `/Workspace/Users/<自分>/databricks-rag-handson/apps/rag-handson-switcher`）を入れ（フォルダーのアイコンから選んでもよい）、②「デプロイ」をクリックする。

![STEP 11](./images/switcher-11.png)


→ 依存パッケージのインストールとビルドは Databricks が行う（数分）。

### STEP 12：アプリを開く
①状態が「実行中」になったら、②アプリの URL をクリックする。

![STEP 12](./images/switcher-12.png)


→ 初めて開くと、アプリが本人に代わって Vector Search・Genie・Model Serving を使うことへの同意の画面が出る。内容を確かめて「許可」をクリックする。

> **注意**：コードを更新したとき（Git フォルダーで Pull したあと）は、「デプロイ」をもう一度クリックする。

## 3-4. 2つのエージェントを比べる

### STEP 13：Genie Agent を指定する
①「Managed」タブを開く。②2 で作った Genie Agent の URL（Genie Agent を開いたときのブラウザーのアドレス欄、`.../genie/rooms/<ID>`）を貼り付け、③「保存」をクリックする。

![STEP 13](./images/switcher-13.png)


→ ID はこのブラウザーに保存される。別の Genie Agent に変えるときは「変更」をクリックする。

### STEP 14：同じ質問を2つのエージェントに投げる
質問の例をクリックするか、質問を入力して「送信」する。①回答に出典が付くことを確かめ、②「Scratch」タブに切り替えて同じ質問を投げ、回答と出典を比べる。

![STEP 14](./images/switcher-14.png)


→ Managed（Agent mode）は回答に数十秒かかることがある。

## 後片付け

ハンズオンが終わったら、Databricks Apps でアプリを開き、右上の「⋮」→「削除」で削除する。アプリは動いている間ずっと課金される（コンピュートのサイズ M で 0.5 DBU/時）。

## まとめ

- Git フォルダーのアプリのフォルダーを、Databricks Apps の画面からデプロイした
- 索引はアプリのリソース、Genie Agent はアプリの画面で指定し、ファイルは編集しなかった
- 同じ質問を Scratch と Managed に投げ、回答と出典を比べた

### 参考リンク
- [Databricks Apps のデプロイ](https://docs.databricks.com/aws/ja/dev-tools/databricks-apps/deploy)
- [アプリのリソース](https://docs.databricks.com/aws/ja/dev-tools/databricks-apps/resources)
- [アプリの認証（ユーザー認証）](https://docs.databricks.com/aws/ja/dev-tools/databricks-apps/auth)
- [AppKit](https://developers.databricks.com/docs/appkit/v0/)